# Data exploration

## 1. Collect your data files

To analyze the output data of pyHiM, you need to collect and merge both kind of files:
- Trace tables
- Localization tables (optionnal)

In [ ]:
import os
from IPython.display import Image

# data_path = input("Please enter the path to the folder containing the data:")
data_path = "/home/xdevos/Repositories/XDevos/traceratops_test"

if not os.path.exists(data_path):
    print("Error: The path does not exist. Please check and try again.")
else:
    print(f"Valid path: {data_path}")

### 1.1. Trace tables

In [ ]:
!mkdir -p {data_path}/dest/raw_traces
!cp {data_path}/*/tracing/data/*3D*mask0_ROI-??.ecsv {data_path}/dest/raw_traces

### 1.2. Localization tables

In [ ]:
!localization_cp_files --files {data_path}/*/localize_3d/data/*barcode.dat --destination_folder {data_path}/dest/raw_localization

## 2. Visualize your data

### 2.1. Pearsons correlation

- Is one ROI significantly different from the others?

In [ ]:
!ls {data_path}/dest/raw_traces/*.ecsv | trace_pearsons --pipe -O {data_path}/dest/raw_traces/

### 2.2. Traces to distance matrix

- What does the structure look like for each ROI? 
- Can we already see a structure emerging?

In [ ]:
!ls {data_path}/dest/raw_traces/*.ecsv | trace_to_matrix --pipe -F {data_path}/dest/raw_traces/

### 2.3. Check number of barcodes and traces

- Do I have all the barcodes I need in each ROI?
- How many traces per ROI?
- Is it highly variable?

In [ ]:
!ls {data_path}/dest/raw_traces/*.ecsv |  trace_stats --pipe

### 2.4. Trace description

This script produce 5 outputs :

---
1. `[tracefile]_trace_statistics.[format]`, **3 histograms of barcode statistics showing**:
    - How many barcodes do I have per trace?
    - If I do not count barcodes with the same number that are repeated in the traces: how many unique barcodes do I have per trace?
    - If I remove the localisations kept for the second sub-plot from my traces used for the first sub-plot, how many different barcodes do I still have in my traces? (there should be zero in the majority of cases)
         - *It is interesting to combine this third sub-plot with the `relative_barcode_frequencies`.*
         - *It would be interesting to know in what type of trace there are repetitions. Is it random? Only traces with few barcodes or, on the contrary, large traces?*
---


2. `[tracefile]_first_neighbor_distances.[format]`, **histograms of distances between consecutive neighboring barcodes**:
    - Are my traces compact or spread out?
    - The histogram profiles should be similar on each axis if we consider that the orientation of the traces is random.
    - *Should they also be symmetrical and Gaussian-shaped?*
    - If the histogram profiles do not appear normal, this means that you will either need to separate the traces (splitter or filter_advanced) or remove the false traces that distort the profile with filters.
    - If the problem is only on the Z axis, it is possible that the alignment is poor on this axis and it can be corrected using `trace_correct_coordinates`.
    - *This plot should be kept for comparison and to visualise any improvement during the analysis.*

---
3. `[tracefile]_barcode_detection.[format]`, **Plot of barcode detection efficiency**:
    - Are there significantly fewer barcodes in the traces?
    - Is this the same for each ROI?
        - If so, this may mean that these barcodes did not work during acquisition.
        - If not, the problem may be with the ROI; compare it with the others.
    - The detection frequency should increase during the analysis as bad traces are removed.
    - *This is therefore a plot to keep for comparison.*

---
4. `[tracefile]_relative_barcode_frequencies`, **Barcode statistics file**:
    - Do I have the same barcode number multiple times in each trace? If so, this may be:
        - A detection issue (filter by intensity);
        - A segmentation issue (splitter or filter_advanced);
        - or a biological reality (filter or splitter on a case-by-case basis).

---
5. `[tracefile]_traces_XYZ.[format]`, **Visual representation of the traces** (if –plotXYZ is set):
    - How are my tracks distributed in space?
    - Will I need to filter the data on the extreme planes in X, Y or Z?


In [ ]:
!ls {data_path}/dest/raw_traces/*.ecsv | trace_analyzer --pipe

### 2.5. Intensity_analyzer

- Do I have the same distribution of intensity values between ROIs? Between barcodes? 
- These values will allow filtering based on intensity.
- *It would be interesting to compare intensities only for repeated barcodes to help find a method for resolving the repetition issue.*

## 3. Remove unusable ROIs

To remove, delete the ecsv trace file.

In [ ]:
!rm {data_path}/dest/raw_traces/Trace_3D_barcode_mask-mask0_ROI-17.ecsv
!rm {data_path}/dest/raw_traces/Trace_3D_barcode_mask-mask0_ROI-19.ecsv
!rm {data_path}/dest/raw_traces/Trace_3D_barcode_mask-mask0_ROI-23.ecsv

## 4. Merge data
Merge the ROIs to be analysed into a single file.

- Trace tables :

In [ ]:
!ls {data_path}/dest/raw_traces/*.ecsv | trace_merge -F {data_path}/dest

- Localization tables :

In [ ]:
!ls {data_path}/dest/raw_localization/*.dat | localization_merge -O {data_path}/dest

## Next tutorial: Analysis steps

*Perform one of the analysis steps and then return to the relevant visualisation scripts to observe an improvement in the data.*